In [3]:
import importlib
import pandas as pd

import load_data
import labels
import build_features
import extract_ontology
import build_topology_graph
import build_ontology
import util

importlib.reload(load_data)
importlib.reload(labels)
importlib.reload(build_features)
importlib.reload(extract_ontology)
importlib.reload(build_topology_graph)
importlib.reload(build_ontology)
importlib.reload(util)

from load_data import load_alerts_from_json
from labels import assign_labels
from util import attacks_per_period_report

### Load data
##### Wazuh + Aminer Alerts (AIT-ADS, .json format) to .csv

In [4]:
output_file = "combined_ait.csv"
dir_path = "../data/ait_ads"

df = load_alerts_from_json(output_file, dir_path)

Opening file ../data/ait_ads/harrison_aminer.json...
Opening file ../data/ait_ads/wardbeck_wazuh.json...
Opening file ../data/ait_ads/wheeler_wazuh.json...
Opening file ../data/ait_ads/shaw_wazuh.json...
Opening file ../data/ait_ads/wilson_aminer.json...
Opening file ../data/ait_ads/fox_aminer.json...
Opening file ../data/ait_ads/santos_wazuh.json...
Opening file ../data/ait_ads/fox_wazuh.json...
Opening file ../data/ait_ads/shaw_aminer.json...
Opening file ../data/ait_ads/wheeler_aminer.json...
Opening file ../data/ait_ads/russellmitchell_aminer.json...
Opening file ../data/ait_ads/harrison_wazuh.json...
Opening file ../data/ait_ads/wilson_wazuh.json...
Opening file ../data/ait_ads/santos_aminer.json...
Opening file ../data/ait_ads/russellmitchell_wazuh.json...
Opening file ../data/ait_ads/wardbeck_aminer.json...
Writing data to combined_ait.csv...

Done.


##### Load .csv to notebook

In [5]:
df = pd.read_csv("../data/ait_ads/combined_ait.csv", low_memory=False)
df.head()

,timestamp,scenario,source,category,entity,raw_log,host_ip,host,rule_id,rule_desc,...,srcport,dstport,proto,is_ids_alert,ids_signature,ids_category,ids_severity,data_json,wazuh_level,alert_id
0,2022-02-04 00:00:01.880000114+00:00,harrison,aminer,AMiner: New event type.,10.237.0.4,type=USER_ACCT msg=audit(1643932801.876:757): ...,10.237.0.4,NaN,NaN,New path(es) detected,...,NaN,NaN,NaN,0,NaN,NaN,NaN,{},NaN,32fd204177e9336ee877f9cfa2e2da48763af614
1,2022-02-04 00:00:01.880000114+00:00,harrison,aminer,AMiner: New user_acct parameter combination in...,10.237.0.4,type=USER_ACCT msg=audit(1643932801.876:757): ...,10.237.0.4,NaN,NaN,New value combination(s) detected,...,NaN,NaN,NaN,0,NaN,NaN,NaN,{},NaN,da63a8ecc82a3475f130cc66a96d932642f4bef9
2,2022-02-04 00:00:01.880000114+00:00,harrison,aminer,AMiner: New event type.,10.237.0.4,type=CRED_ACQ msg=audit(1643932801.876:758): p...,10.237.0.4,NaN,NaN,New path(es) detected,...,NaN,NaN,NaN,0,NaN,NaN,NaN,{},NaN,4aadf38ab0ade7b66368f6446bc086e26568a5a4
3,2022-02-04 00:00:01.880000114+00:00,harrison,aminer,AMiner: New cred_acq parameter combination in ...,10.237.0.4,type=CRED_ACQ msg=audit(1643932801.876:758): p...,10.237.0.4,NaN,NaN,New value combination(s) detected,...,NaN,NaN,NaN,0,NaN,NaN,NaN,{},NaN,facbffaac81011d2052cf6a1ff69b601e701cc70
4,2022-02-04 00:00:01.880000114+00:00,harrison,aminer,AMiner: New event type.,10.237.0.4,type=LOGIN msg=audit(1643932801.876:759): pid=...,10.237.0.4,NaN,NaN,New path(es) detected,...,NaN,NaN,NaN,0,NaN,NaN,NaN,{},NaN,28a9d415d0fe568d65d135e8289b75f925ff3861


#### Sanity checks

In [7]:
print("Min timestamp in dataset: ", df["timestamp"].min(), "\nMax timestamp in dataset: ", df["timestamp"].max())
df.columns

Min timestamp in dataset:  2022-01-14 00:00:01.750000+00:00 
Max timestamp in dataset:  2022-02-08 23:59:46.130000114+00:00


Index(['timestamp', 'scenario', 'source', 'category', 'entity', 'raw_log',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'alert_channel', 'decoder', 'decoder_parent', 'location',
       'mitre_ids', 'mitre_tactic', 'mitre_technique', 'username', 'procname',
       'aminer_component_type', 'aminer_training_mode', 'aminer_new_event',
       'srcip', 'dstip', 'srcport', 'dstport', 'proto', 'is_ids_alert',
       'ids_signature', 'ids_category', 'ids_severity', 'data_json',
       'wazuh_level', 'alert_id'],
      dtype='object')

## Labelling

In [8]:
assign_labels(df)

Labeling dataset...
Done. Wrote labeled dataset to: ../data/ait_ads/labeled_combined_ait.csv


### Attack report

In [9]:
labeled_df = pd.read_csv("../data/ait_ads/labeled_combined_ait.csv", low_memory=False)

In [10]:
labeled_df.columns

Index(['timestamp', 'scenario', 'source', 'category', 'entity', 'raw_log',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'alert_channel', 'decoder', 'decoder_parent', 'location',
       'mitre_ids', 'mitre_tactic', 'mitre_technique', 'username', 'procname',
       'aminer_component_type', 'aminer_training_mode', 'aminer_new_event',
       'srcip', 'dstip', 'srcport', 'dstport', 'proto', 'is_ids_alert',
       'ids_signature', 'ids_category', 'ids_severity', 'data_json',
       'wazuh_level', 'alert_id', 'event_label', 'y', 'attack_type'],
      dtype='object')

In [11]:
# number of attakcs in the dataset for each scenario
print(labeled_df["y"].value_counts())

labeled_df[labeled_df["y"] == 1]["scenario"].value_counts()
# or
labeled_df.groupby("scenario")["y"].sum().sort_values(ascending=False)

print(labeled_df["event_label"].value_counts())
print(labeled_df.groupby("scenario")["event_label"].value_counts())

overall, by_scn, summary = attacks_per_period_report(
    labeled_df,
    period="day",                  # "hour" | "day" | "week"
    scenario_col="scenario",
    tz="UTC",
    week_start="MON",
)

print(summary)
display(overall.head(10))
display(by_scn[by_scn["has_both_classes"]].head(50))

y
0    2349042
1     306779
Name: count, dtype: int64
event_label
benign                         2349042
attack:dirb                     249226
attack:wpscan                    54570
attack:cracking                   1987
attack:service_scans               629
attack:dnsteal                     240
attack:privilege_escalation        108
attack:webshell                     18
attack:reverse_shell                 1
Name: count, dtype: int64
scenario         event_label                
fox              benign                         401076
                 attack:dirb                     62305
                 attack:wpscan                    9474
                 attack:cracking                   168
                 attack:service_scans               63
                 attack:dnsteal                      9
                 attack:privilege_escalation         7
                 attack:webshell                     2
harrison         benign                         521060
                 

,bucket,n_total,n_attacks,n_benign,attack_rate,has_both_classes
0,2022-01-14 00:00:00+00:00,32056,0,32056,0.000000,False
1,2022-01-15 00:00:00+00:00,36752,0,36752,0.000000,False
2,2022-01-16 00:00:00+00:00,20477,0,20477,0.000000,False
3,2022-01-17 00:00:00+00:00,12759,62,12697,0.004859,True
4,2022-01-18 00:00:00+00:00,5659,149,5510,0.026330,True
5,2022-01-19 00:00:00+00:00,17520,0,17520,0.000000,False
6,2022-01-20 00:00:00+00:00,6581,0,6581,0.000000,False
7,2022-01-21 00:00:00+00:00,18912,0,18912,0.000000,False
8,2022-01-22 00:00:00+00:00,12489,0,12489,0.000000,False
9,2022-01-23 00:00:00+00:00,13147,33,13114,0.002510,True


,scenario,bucket,n_total,n_attacks,n_benign,attack_rate,has_both_classes
2,fox,2022-01-17 00:00:00+00:00,3843,2,3841,0.000520,True
3,fox,2022-01-18 00:00:00+00:00,5659,149,5510,0.026330,True
9,harrison,2022-02-08 00:00:00+00:00,19227,354,18873,0.018412,True
13,russellmitchell,2022-01-24 00:00:00+00:00,4636,76,4560,0.016393,True
17,santos,2022-01-17 00:00:00+00:00,8916,60,8856,0.006729,True
22,shaw,2022-01-29 00:00:00+00:00,4211,14,4197,0.003325,True
28,wardbeck,2022-01-23 00:00:00+00:00,7669,33,7636,0.004303,True
32,wheeler,2022-01-29 00:00:00+00:00,28238,222,28016,0.007862,True
33,wheeler,2022-01-30 00:00:00+00:00,29731,160,29571,0.005382,True
38,wilson,2022-02-07 00:00:00+00:00,18552,165,18387,0.008894,True


Check start and end times per scenario